# The MPC's model with bristle friction (100 µm), held along the plan

A model for the MPC that predicts sliding. It keeps the plant's bristle friction (`RobotSimulationOpenLoopModel_Merged.ipynb`, Steps 5.5 and 5.6): the force follows the bristle deflection $z$, so a wheel can hold force while gripping and break loose into sliding. Built from the plant's `simulation_parameters`.

* **State** $\mathbf{x} = [X, Y, \theta, v_x, v_y, \dot{\theta}, \omega_1..\omega_4, z_1..z_4]$: the pose, the velocity along the robot's own axes, the wheel speeds (measured) and the bristle deflections (estimated). **Input** $\mathbf{u}$: the four motor voltages.
* **Softer bristle than the plant:** $\delta = 100\ \mu m$ (the plant keeps $2\ \mu m$). The motor current settles at once.
* **Held values make it linear:** the bristle relax rate $d_i$, the normal forces $N_i$, the rolling drag, the roller factor and the turning rate are held for each step of the horizon, at the state the last plan predicted for that step. With those held, the model is exactly linear, so it is discretized exactly over $T_s$ with one matrix exponential. **No forward Euler and no sub-steps.**
* Because each step of the horizon has its own $d_i$, the prediction can follow a wheel going from gripping to sliding within the 0.4 s.

In [ ]:
# ===========================================================================
# The plant's parameters and robot class (the model takes its geometry and values from them)
# ===========================================================================
import json
import numpy as np
from scipy.linalg import expm

PLANT_NOTEBOOK = "RobotSimulationOpenLoopModel_Merged.ipynb"
plant_cells = {c.get("id"): "".join(c["source"]) for c in json.load(open(PLANT_NOTEBOOK, encoding="utf-8"))["cells"]}
exec(plant_cells["constants"])     # simulation_parameters, GRAVITY
exec(plant_cells["plant"])         # OmniMotorState, OmniRobotSimulation
print(f"Loaded the parameters and robot class from {PLANT_NOTEBOOK}")

## Equations

<details>
<summary><b> Open this for the equations</b></summary>

**Slip** (from the state): $v_{s,i} = (\mathbf{H}_{drive}\mathbf{v})_i - r\,\omega_i$.

**Bristle** (the plant's Step 5.5, with $\delta = 100\ \mu m$): $\delta\, \dot{z}_i = v_{s,i} - d_i\, z_i$, with the relax rate $d_i = \frac{\mu_s}{\mu_{d,i}}\sqrt{v_{s,i}^2 + v_c^2}$ **held**. A small $d_i$ means gripping (the bristle holds its deflection, like a spring); a large $d_i$ means sliding (the bristle settles at once to its sliding value).

**Friction** (Step 5.6): $F_i = -\mu_s N_i z_i$, with the normal force $N_i$ **held**.

**What $z$ means:** $|z_i| = |F_i| / (\mu_s N_i)$, the share of the wheel's static grip limit $\mu_s N_i$ in use. The friction coefficient being used is $|F_i| / N_i = \mu_s |z_i|$:

| $\lvert z_i \rvert$ | friction used, $\lvert F_i \rvert / N_i$ | meaning |
|---|---|---|
| 0 | 0 | no friction force |
| 0.9 | 0.81 | the MPC's bound, $\mu_s - 3\sigma_\mu$ (see `RobotSimulationMPC.ipynb`) |
| 1 | 0.9 | the full static limit $\mu_s N_i$ |

$z$ says how much friction a wheel uses, **not** whether it slides: a sliding wheel settles at $|z_i| = \mu_{d,i} / \mu_s$, which is below 1 (for example about 0.79 at 0.5 m/s of slip). Whether a wheel slides is the slip $v_{s,i}$.

**Wheel** (Step 5.7, current settled): $J_w \dot{\omega}_i = K_t \frac{u_i - K_b \omega_i}{R} - c_i N_i r - r F_i$, with the rolling drag factor $c_i = C_{rr}\tanh(\omega_i/\varepsilon)$ **held**.

**Chassis:** $\mathbf{M}\mathbf{a} = \mathbf{H}_{drive}^\top \mathbf{F} + \mathbf{H}_{axial}^\top (\boldsymbol{\phi}_a \circ \mathbf{N})$ (roller factor $\phi_{a,i}$ **held**), $\frac{d\mathbf{v}}{dt} = \mathbf{a} + \bar{\dot{\theta}}\,\mathbf{S}\mathbf{v}$ (turning rate **held**).

**Normal forces** at a state: the plant's load transfer (Section 4) with this step's acceleration, $\mathbf{N} = \mathbf{N}_{\text{static}} + \mathbf{T}\mathbf{a}$, solved together with the forces.

**Pose:** the distance moved along the robot's axes over the step, turned into the room frame at the step's heading.

With the held values, $\frac{d}{dt}[\mathbf{v}, \boldsymbol{\omega}, \mathbf{z}] = \mathbf{A}\,[\mathbf{v}, \boldsymbol{\omega}, \mathbf{z}] + \mathbf{B}\mathbf{u} + \mathbf{c}$ is linear, and is held over $T_s$ exactly (zero-order hold): one matrix exponential per step of the horizon.

**Where the held values come from:**
* In the MPC: at each step $j$ of the horizon, the state the **last plan** predicted for that step (moved on one step); step 0 uses the estimate. Within one QP they are constants, so the problem stays a convex QP.
* In the prediction check below: the state the model itself reaches at each step.

**Bristle estimate:** $z$ cannot be measured. Each MPC step it is estimated from the wheel's torque balance over the last $T_s$: $F_i = \left(K_t i_i - c_i N_i r - J_w \Delta\omega_i / T_s\right) / r$ and $z_i = -F_i / (\mu_s N_i)$, using the measured wheel speeds and the voltage that was applied.

</details>

In [ ]:
"""
The MPC's model with bristle friction
=====================================

BristleModel is the robot as this MPC sees it: the plant's bristle friction (Steps 5.5 and 5.6), with a softer
bristle (100 um against the plant's 2 um) and the motor current settled at once. Five values that change slowly
are held for each step of the horizon (the bristle relax rate, the normal forces, the rolling drag, the roller
force and the turning rate). With them held, the model is linear, so it is held over a whole controller step
exactly with one matrix exponential: no forward Euler and no sub-steps.

State x = [X, Y, theta, v_x, v_y, theta_dot, omega_1..4, z_1..4]: the pose, the velocity along the robot's own
axes, the wheel speeds and the bristle deflections. Input u: the four motor voltages.
The part that moves with the model's equations is w = [v, omega, z] = x[3:14].

The controller calls:
    held_values(x)                   the values held for one step, at the state x
    linear_model(held)               dw/dt = A w + B u + c with those values
    discretize(As, Bs, cs, Ts)       several steps' models held over Ts at once
"""


class BristleModel:
    """
    The robot with bristle friction, for the MPC. Built from simulation_parameters, the same values as the plant.
    """

    def __init__(self, params: dict, deflection: float = 100e-6):
        """
        params: simulation_parameters
        deflection: the model's bristle deflection delta [m] (the plant uses lugre_deflection, 2 um)
        """
        self.p = params
        self.delta = deflection

        # -------------------------------------------------------------------
        # Fixed geometry and loads, worked out by the plant's robot class from the same parameters.
        # Only fixed values are read from it, never a state.
        # -------------------------------------------------------------------
        geometry = OmniRobotSimulation(params)

        # Drive speeds of the wheels = H_drive v, axle (roller) speeds = H_axial v
        self.H = geometry.H_drive
        self.H_a = geometry.H_axial

        # 1 / mass and 1 / yaw inertia, for [x, y, yaw]
        self.M_inv = np.diag(geometry.M_inv)

        # Normal force on each wheel at rest [N]
        self.F_n_static = np.array([geometry.F_n_front_static, geometry.F_n_front_static,
                                    geometry.F_n_rear_static, geometry.F_n_rear_static])

        # Load transfer (Section 4) as a matrix: F_n = F_n_static + T_load a, with a = [right, forward, yaw]
        pitch = geometry.pitch_gain
        roll_front = geometry.roll_gain_front
        roll_rear = geometry.roll_gain_rear
        self.T_load = np.array([
            [-roll_front, -pitch, 0.0],     # 1 FR
            [roll_front, -pitch, 0.0],      # 2 FL
            [roll_rear, pitch, 0.0],        # 3 RL
            [-roll_rear, pitch, 0.0],       # 4 RR
        ])

        # Turning of the robot's own axes: dv/dt = a + theta_dot S v
        self.S = np.array([[0.0, 1.0, 0.0],
                           [-1.0, 0.0, 0.0],
                           [0.0, 0.0, 0.0]])

    # =======================================================================
    # Values held for one step
    # =======================================================================

    def held_values(self, x: np.ndarray) -> tuple:
        """
        The five values held for one step of the horizon, worked out at the state x.

        x: state [X, Y, theta, v_x, v_y, theta_dot, omega_1..4, z_1..4]

        Returns (d, F_n, c_roll, phi_a, turn):
            d: bristle relax rate of each wheel [m/s]: small while gripping, large while sliding (Step 5.5)
            F_n: normal force on each wheel, with this state's acceleration (Section 4) [N]
            c_roll: rolling drag per unit normal force (Step 5.3)
            phi_a: roller force per unit normal force (Step 5.8)
            turn: turning rate theta_dot [rad/s]
        """
        p = self.p
        v = x[3:6]
        omega = x[6:10]
        z = x[10:14]

        # Drive slip of each wheel: v_slip_d = H_drive v - r omega
        v_slip_d = self.H @ v - p['r'] * omega

        # Bristle relax rate (Steps 5.4 and 5.5): d = (mu_s / mu_d) sqrt(v_slip_d^2 + v_c^2)
        mu_d = p['mu_k'] + (p['mu_s'] - p['mu_k']) * np.exp(-p['alpha_d'] * np.abs(v_slip_d))
        d = (p['mu_s'] / mu_d) * np.sqrt(v_slip_d**2 + p['lugre_creep']**2)

        # Roller force per unit normal force, from the axle slip v_slip_a = H_axial v
        if p['roller_friction']:
            v_slip_a = self.H_a @ v
            mu_a = p['mu_a_k'] + (p['mu_a_s'] - p['mu_a_k']) * np.exp(-p['alpha_a'] * np.abs(v_slip_a))
            phi_a = -mu_a * (2.0 / np.pi) * np.arctan(p['roller_smoothing'] * v_slip_a)
        else:
            phi_a = np.zeros(4)

        # Normal forces with this state's acceleration. The floor forces are each a factor times their wheel's
        # normal force, F = -mu_s F_n z and F_a = phi_a F_n, so
        #   a = M^-1 K F_n,   K = -mu_s H_drive^T diag(z) + H_axial^T diag(phi_a)
        # and with F_n = F_n_static + T_load a:
        #   (I - T_load M^-1 K) F_n = F_n_static
        K = -p['mu_s'] * self.H.T * z + self.H_a.T * phi_a
        F_n = np.linalg.solve(np.eye(4) - self.T_load @ (self.M_inv[:, None] * K), self.F_n_static)
        F_n = np.maximum(0.0, F_n)              # a wheel cannot pull on the floor

        # Rolling drag per unit normal force
        c_roll = p['Crr'] * np.tanh(omega / p['roll_smoothing'])

        turn = v[2]
        return d, F_n, c_roll, phi_a, turn

    # =======================================================================
    # The model as matrices
    # =======================================================================

    def linear_model(self, held: tuple) -> tuple:
        """
        The model with the held values: dw/dt = A w + B u + c, for w = [v (3), omega (4), z (4)].

        held: from held_values

        Returns (A, B, c), shapes (11, 11), (11, 4), (11,).
        """
        p = self.p
        d, F_n, c_roll, phi_a, turn = held
        r = p['r']
        R = p['R']
        Kt = p['Kt']
        Kb = p['Kb']
        Jw = p['Jw']
        mu_s = p['mu_s']

        A = np.zeros((11, 11))
        B = np.zeros((11, 4))
        c = np.zeros(11)

        # Rows 0-2, chassis (Step 5.9), along the robot's axes:
        #   dv/dt = M^-1 (H_drive^T F + H_axial^T F_a) + turn S v,   F = -mu_s F_n z,   F_a = phi_a F_n
        A[0:3, 0:3] = turn * self.S
        A[0:3, 7:11] = -mu_s * self.M_inv[:, None] * (self.H.T * F_n)
        c[0:3] = self.M_inv * (self.H_a.T @ (phi_a * F_n))

        # Rows 3-6, wheels (Step 5.7), with the settled current I = (u - Kb omega) / R:
        #   Jw domega/dt = Kt I - c_roll F_n r - r F = Kt/R u - Kt Kb/R omega - c_roll F_n r + r mu_s F_n z
        A[3:7, 3:7] = -Kt * Kb / (R * Jw) * np.eye(4)
        A[3:7, 7:11] = np.diag(r * mu_s * F_n / Jw)
        B[3:7, 0:4] = Kt / (R * Jw) * np.eye(4)
        c[3:7] = -c_roll * F_n * r / Jw

        # Rows 7-10, bristles (Step 5.5):
        #   delta dz/dt = v_slip_d - d z = H_drive v - r omega - d z
        A[7:11, 0:3] = self.H / self.delta
        A[7:11, 3:7] = -r * np.eye(4) / self.delta
        A[7:11, 7:11] = -np.diag(d) / self.delta

        return A, B, c

    # =======================================================================
    # Controller steps
    # =======================================================================

    @staticmethod
    def discretize(As: np.ndarray, Bs: np.ndarray, cs: np.ndarray, Ts: float) -> tuple:
        """
        K steps' models, each held over one controller step Ts with the voltages held (zero-order hold).
        Exact for each model. Also gives the distance moved along the robot's axes, s, which moves the pose.

        As, Bs, cs: K models from linear_model, shapes (K, 11, 11), (K, 11, 4), (K, 11)
        Ts: controller step [s]

        Returns (A_d, B_d, dd, S_w, S_u, s_0), each with K entries:
            w_next = A_d w + B_d u + dd
            s      = S_w w + S_u u + s_0
        """
        K = len(As)

        # w, the distance moved s (ds/dt = v, the first three entries of w), the held voltages u and a
        # constant 1 go into one matrix, so that one matrix exponential holds the whole step:
        #
        #   d/dt [w, s, u, 1] = [[A, 0, B, c],     w (11)
        #                        [V, 0, 0, 0],     s (3), V picks v out of w
        #                        [0, 0, 0, 0],     u (4, held)
        #                        [0, 0, 0, 0]]     1
        block = np.zeros((K, 19, 19))
        block[:, 0:11, 0:11] = As
        block[:, 0:11, 14:18] = Bs
        block[:, 0:11, 18] = cs
        block[:, 11:14, 0:3] = np.eye(3)
        E = expm(block * Ts)

        A_d = E[:, 0:11, 0:11]
        B_d = E[:, 0:11, 14:18]
        dd = E[:, 0:11, 18]
        S_w = E[:, 11:14, 0:11]
        S_u = E[:, 11:14, 14:18]
        s_0 = E[:, 11:14, 18]
        return A_d, B_d, dd, S_w, S_u, s_0

    def predict(self, x0: np.ndarray, U: np.ndarray, Ts: float) -> np.ndarray:
        """
        Prediction from x0 with the voltages U, one controller step at a time. The values are held at each state
        the model reaches, so a wheel can go from gripping to sliding during the prediction.

        x0: state now (14,)
        U: voltages for each step (N, 4) [V]
        Ts: controller step [s]

        Returns the predicted states after each step (N, 14).
        """
        states = []
        x = x0.copy()
        for u in U:
            # This step's model, held over Ts
            A, B, c = self.linear_model(self.held_values(x))
            A_d, B_d, dd, S_w, S_u, s_0 = (item[0] for item in self.discretize(A[None], B[None], c[None], Ts))

            # Velocity, wheel speeds and bristles at the end of the step, and the distance moved
            w = x[3:14]
            w_next = A_d @ w + B_d @ u + dd
            s = S_w @ w + S_u @ u + s_0

            # Pose: the distance moved along the robot's axes, turned into the room frame at this step's heading
            cos_th, sin_th = np.cos(x[2]), np.sin(x[2])
            pose_next = x[0:3] + np.array([cos_th * s[0] - sin_th * s[1],
                                           sin_th * s[0] + cos_th * s[1],
                                           s[2]])

            x = np.concatenate((pose_next, w_next))
            states.append(x)
        return np.array(states)

### Check: does it predict sliding?

From the plant's own states during manoeuvres that slide, each model predicts 0.4 s ahead (20 steps of 20 ms) with the voltages the plant actually got, and the position is compared with the plant's. The no-slip model of `MPCModel_GripLimit.ipynb` is shown alongside. The plant runs with its noise off so only the model differs. The bristle model starts from the plant's true $z$ here (in the MPC it is estimated).

In [ ]:
# ===========================================================================
# Check: prediction against the plant, 0.4 s ahead, in manoeuvres that slide
# ===========================================================================
# The no-slip model of MPCModel_GripLimit.ipynb, for comparison
grip_cells = {c.get("id"): "".join(c["source"]) for c in json.load(open("MPCModel_GripLimit.ipynb", encoding="utf-8"))["cells"]}
exec(grip_cells["mpc_model"])      # AffineGripModel

Ts = 0.02                          # controller step [s]
N_steps = 20                       # prediction length: 20 steps = 0.4 s
P0 = dict(simulation_parameters, sigma_V=0.0, sigma_mu=0.0)     # plant noise off, so only the model differs
bristle = BristleModel(P0)
no_slip = AffineGripModel(P0)

FWD = np.array([1.0, -1.0, -1.0, 1.0])     # wheel voltage signs: forward
STR = np.array([-1.0, -1.0, 1.0, 1.0])     # wheel voltage signs: strafe right
stage = lambda t: 1.0 if t < 1 else (-1.0 if t < 2 else 0.0)    # drive, reverse, coast
random_V = np.random.default_rng(3).uniform(-8, 8, (20, 4))

# (name, voltages as a function of time [V], run length [s])
manoeuvres = [
    ("10 V strafe", lambda t: 10 * STR, 2.0),
    ("4 V fwd/back/coast", lambda t: 4 * FWD * stage(t), 3.0),
    ("4 V side-to-side", lambda t: -4 * STR * stage(t), 3.0),
    ("random 8 V steps", lambda t: random_V[min(int(t / 0.2), 19)], 4.0),
]


def plant_run(V_of_t, T):
    """The plant, with each voltage held for Ts. Returns its state at the start of every step and the voltages."""
    robot = OmniRobotSimulation(P0)
    state = OmniMotorState()
    states, voltages = [], []
    for j in range(int(round(T / Ts))):
        V = V_of_t(j * Ts)
        states.append(np.concatenate((state.pos_global, state.v_local, state.omega, state.z)))
        voltages.append(V)
        for _ in range(int(round(Ts / P0['dt']))):
            state, _, _ = robot.step(state, V)
    states.append(np.concatenate((state.pos_global, state.v_local, state.omega, state.z)))
    return np.array(states), np.array(voltages)


def no_slip_predict(x0, U):
    """Prediction with the no-slip model, holding its values at each state it reaches (as the bristle model does)."""
    pose = x0[0:3].copy()
    v = x0[3:6].copy()
    states = []
    for u in U:
        A_d, B_d, d, S_v, S_u, s_0 = no_slip.discretize(no_slip.affine_model(*no_slip.held_values(v)), Ts)
        s = S_v @ v + S_u @ u + s_0
        v = A_d @ v + B_d @ u + d
        cos_th, sin_th = np.cos(pose[2]), np.sin(pose[2])
        pose = pose + np.array([cos_th * s[0] - sin_th * s[1], sin_th * s[0] + cos_th * s[1], s[2]])
        states.append(np.concatenate((pose, v)))
    return np.array(states)


print("Position error after 0.4 s [mm], median (90th percentile)")
for name, V_of_t, T in manoeuvres:
    X, U = plant_run(V_of_t, T)

    # How much of the time the plant's wheels slide (drive slip over 5 mm/s)
    v_slip_d = X[:, 3:6] @ bristle.H.T - P0['r'] * X[:, 6:10]
    sliding = 100 * np.mean(np.abs(v_slip_d) > 5e-3)

    # From every 5th step, both models predict 0.4 s ahead with the plant's voltages (bristle model from the true z)
    errors = {"bristle": [], "no-slip": []}
    for k in range(5, len(U) - N_steps, 5):
        truth = X[k + N_steps, 0:2]
        errors["bristle"].append(1e3 * np.hypot(*(bristle.predict(X[k], U[k:k + N_steps], Ts)[-1, 0:2] - truth)))
        errors["no-slip"].append(1e3 * np.hypot(*(no_slip_predict(X[k], U[k:k + N_steps])[-1, 0:2] - truth)))

    results = "   ".join(f"{model}: {np.median(e):6.1f} ({np.percentile(e, 90):6.1f})" for model, e in errors.items())
    print(f"   {name:20} (wheels sliding {sliding:3.0f}% of the time)   {results}")